# 🧬 Module 14 — OOP Part 2: Inheritance, Polymorphism, Abstraction & Design

## 🎯 What you will learn
1. Inheritance: single, multilevel, hierarchical, multiple, hybrid
2. `super()`, method overriding, extending `__init__`
3. The Method Resolution Order (MRO) and the diamond problem
4. Polymorphism & duck typing
5. Abstraction with `abc` (abstract base classes) and Protocols
6. Composition vs inheritance ("has-a" vs "is-a")
7. Mixins, `isinstance`/`issubclass`
8. Design patterns you'll meet: Singleton, Factory, Observer, Strategy
9. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐

---
## 1. Inheritance — reuse and specialise
A **child (derived / subclass)** inherits attributes and methods of a **parent (base / superclass)** and can add or override.
Use it for **"is-a"** relationships: a Dog *is an* Animal.

In [1]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

    def eat(self):
        return f"{self.name} is eating"

class Dog(Animal):                       # Dog inherits from Animal
    def speak(self):                     # override
        return f"{self.name} says Woof"

class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow"

class Fish(Animal):
    pass                                 # inherits everything unchanged

for pet in (Dog("Rex"), Cat("Tom"), Fish("Nemo")):
    print(pet.speak(), "|", pet.eat())

print(issubclass(Dog, Animal), isinstance(Dog("x"), Animal), isinstance(Animal("y"), Dog))
print(Dog.__bases__, Dog.__mro__)

Rex says Woof | Rex is eating
Tom says Meow | Tom is eating
Nemo makes a sound | Nemo is eating
True True False
(<class '__main__.Animal'>,) (<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


## 2. `super()` — call the parent

In [2]:
class Employee:
    def __init__(self, name, salary):
        self.name, self.salary = name, salary

    def annual(self):
        return self.salary * 12

    def __str__(self):
        return f"{self.name} earns {self.salary}/month"

class Manager(Employee):
    def __init__(self, name, salary, team):
        super().__init__(name, salary)   # reuse the parent's initialiser
        self.team = team                 # add new stuff

    def annual(self):                    # extend behaviour, don't duplicate it
        return super().annual() + 5000   # bonus

    def __str__(self):
        return super().__str__() + f" and manages {len(self.team)} people"

m = Manager("Asha", 5000, ["Ravi", "Meena"])
print(m, "|", m.annual())

Asha earns 5000/month and manages 2 people | 65000


> ⚠️ If you override `__init__` and forget `super().__init__()`, the parent's attributes are never created!

In [3]:
class Bad(Employee):
    def __init__(self, extra):
        self.extra = extra               # forgot super().__init__
try:
    Bad(1).annual()
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'Bad' object has no attribute 'salary'


## 3. Types of inheritance

```
Single:      A → B
Multilevel:  A → B → C
Hierarchical:  A → B, A → C, A → D   (one parent, many children)
Multiple:    A, B → C                (many parents)
Hybrid:      a mix (diamond: A → B, A → C, B & C → D)
```

In [4]:
# Multilevel
class Vehicle:
    def start(self): return "engine on"
class Car(Vehicle):
    def wheels(self): return 4
class SportsCar(Car):
    def turbo(self): return "boost!"
s = SportsCar()
print(s.start(), s.wheels(), s.turbo())

# Multiple inheritance
class Flyer:
    def move(self): return "flying"
    def altitude(self): return 1000
class Swimmer:
    def move(self): return "swimming"
class Duck(Flyer, Swimmer):              # order matters!
    pass
d = Duck()
print(d.move(), d.altitude())            # Flyer's move wins (listed first)
print([c.__name__ for c in Duck.__mro__])

engine on 4 boost!
flying 1000
['Duck', 'Flyer', 'Swimmer', 'object']


## 4. Method Resolution Order (MRO) & the diamond problem

Python computes a linear order of classes to search for a method using **C3 linearization**:
1. children before parents
2. left parent before right parent (in the class statement)
3. each class appears once

In [5]:
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B > " + super().who()
class C(A):
    def who(self): return "C > " + super().who()
class D(B, C):
    def who(self): return "D > " + super().who()

print(D().who())                         # D > B > C > A   (super follows the MRO, not just the parent)
print([k.__name__ for k in D.mro()])

# Inconsistent hierarchies are rejected
try:
    class X(A, B): pass
except TypeError as e:
    print("TypeError:", e)

D > B > C > A
['D', 'B', 'C', 'A', 'object']
TypeError: Cannot create a consistent method resolution order (MRO) for bases A, B


> `super()` doesn't mean "my parent" — it means **"the next class in the MRO"**. This is what makes cooperative multiple inheritance work.

## 5. Polymorphism — "many forms"
The same call, different behaviour depending on the object.

### 5.1 Through inheritance (method overriding)

In [6]:
class Shape:
    def area(self): raise NotImplementedError("subclass must implement area()")
    def describe(self): return f"{type(self).__name__} with area {self.area():.2f}"

class Rect(Shape):
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

class Square(Rect):
    def __init__(self, s): super().__init__(s, s)

for sh in (Rect(2, 3), Circle(1), Square(4)):
    print(sh.describe())

total = sum(s.area() for s in (Rect(2, 3), Circle(1), Square(4)))
print(round(total, 2))

Rect with area 6.00
Circle with area 3.14
Square with area 16.00
25.14


### 5.2 Duck typing — "If it walks like a duck and quacks like a duck…"
Python cares about **what an object can do**, not its class. No shared parent needed.

In [7]:
class Duck:
    def speak(self): return "Quack"
class Robot:
    def speak(self): return "Beep"
class Person:
    def speak(self): return "Hello"

def make_it_speak(thing):
    return thing.speak()                 # any object with .speak() works

print([make_it_speak(t) for t in (Duck(), Robot(), Person())])

# Built-ins are polymorphic: len() works on anything with __len__
print(len("abc"), len([1, 2]), len({1: 2}), len(range(5)))
print(1 + 2, "a" + "b", [1] + [2])        # + is polymorphic

['Quack', 'Beep', 'Hello']
3 2 1 5
3 ab [1, 2]


### 5.3 Operator overloading is polymorphism too (see Module 13). Method overloading (same name, different arguments) does not exist — use default args/`*args`, or `singledispatchmethod`:

In [8]:
class Calc:
    def add(self, a, b=0, c=0):
        return a + b + c
print(Calc().add(1), Calc().add(1, 2), Calc().add(1, 2, 3))

from functools import singledispatchmethod
class Printer:
    @singledispatchmethod
    def show(self, x): return f"other: {x}"
    @show.register
    def _(self, x: int): return f"int: {x}"
    @show.register
    def _(self, x: str): return f"str: {x!r}"
print(Printer().show(3), Printer().show("a"), Printer().show(2.5))

1 3 6
int: 3 str: 'a' other: 2.5


## 6. Abstraction — `abc` module
An **abstract base class (ABC)** defines a *contract*: subclasses **must** implement the abstract methods. You cannot instantiate an ABC directly.

In [9]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount): ...

    @abstractmethod
    def refund(self, amount): ...

    def receipt(self, amount):                 # concrete (shared) method
        return f"Paid {amount} via {type(self).__name__}"

class Card(PaymentMethod):
    def pay(self, amount): return f"card charged {amount}"
    def refund(self, amount): return f"card refunded {amount}"

class Cash(PaymentMethod):
    def pay(self, amount): return "cash taken"
    # forgot refund!

try:
    PaymentMethod()
except TypeError as e:
    print("TypeError:", e)
try:
    Cash()
except TypeError as e:
    print("TypeError:", e)

c = Card()
print(c.pay(50), "|", c.receipt(50), "|", isinstance(c, PaymentMethod))

TypeError: Can't instantiate abstract class PaymentMethod without an implementation for abstract methods 'pay', 'refund'
TypeError: Can't instantiate abstract class Cash without an implementation for abstract method 'refund'
card charged 50 | Paid 50 via Card | True


### Protocols (structural typing) — duck typing with type checking

In [10]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Speaker(Protocol):
    def speak(self) -> str: ...

print(isinstance(Duck(), Speaker), isinstance(42, Speaker))   # no inheritance needed!

True False


## 7. Composition vs Inheritance
- **Inheritance** = "is-a" (Car *is a* Vehicle).
- **Composition** = "has-a" (Car *has an* Engine). Prefer it when the relationship isn't truly "is-a": it's more flexible and avoids fragile hierarchies.

> Rule of thumb: **favour composition over inheritance.**

In [11]:
class Engine:
    def __init__(self, hp): self.hp = hp
    def start(self): return f"{self.hp}hp engine running"

class Wheel:
    pass

class Car:
    def __init__(self, engine):
        self.engine = engine                   # HAS-A
        self.wheels = [Wheel() for _ in range(4)]
    def drive(self):
        return f"Car: {self.engine.start()} on {len(self.wheels)} wheels"

print(Car(Engine(150)).drive())
print(Car(Engine(300)).drive())               # swap parts freely

# Aggregation: parts exist independently; Composition: parts die with the owner
class Department:
    def __init__(self, name): self.name, self.employees = name, []
    def hire(self, emp): self.employees.append(emp)

Car: 150hp engine running on 4 wheels
Car: 300hp engine running on 4 wheels


## 8. Mixins — reusable bundles of behaviour
A **mixin** is a small class meant to be combined via multiple inheritance to add one capability.

In [12]:
import json

class JsonMixin:
    def to_json(self):
        return json.dumps(self.__dict__)

class ReprMixin:
    def __repr__(self):
        args = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{type(self).__name__}({args})"

class User(JsonMixin, ReprMixin):
    def __init__(self, name, age): self.name, self.age = name, age

u = User("Asha", 19)
print(u.to_json(), u)

{"name": "Asha", "age": 19} User(name='Asha', age=19)


## 9. `isinstance`, `issubclass`, `type`, `super()` cheat sheet

In [13]:
class P: pass
class Q(P): pass
q = Q()
print(isinstance(q, Q), isinstance(q, P), type(q) == P, type(q) is Q)
print(issubclass(Q, P), issubclass(Q, Q), issubclass(bool, int))
print(isinstance(5, (int, float)), isinstance("a", (int, float)))
print(Q.__bases__, Q.__name__, q.__class__.__mro__[1].__name__)
print(hasattr(q, "__init__"), callable(q), callable(Q))

True True False True
True True True
True False
(<class '__main__.P'>,) Q P
True False True


## 10. Design patterns in Python (short & practical)

In [14]:
# 1) Singleton: exactly one instance
class Config:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.settings = {}
        return cls._instance
print(Config() is Config())

# 2) Factory: create objects without exposing the class choice
class Circle2:  kind = "circle"
class Square2:  kind = "square"
def shape_factory(name):
    return {"circle": Circle2, "square": Square2}[name]()
print(shape_factory("square").kind)

# 3) Strategy: swap algorithms at run time (functions are first-class!)
def discount_none(p): return p
def discount_student(p): return p * 0.8
class Checkout:
    def __init__(self, strategy=discount_none): self.strategy = strategy
    def total(self, price): return self.strategy(price)
print(Checkout().total(100), Checkout(discount_student).total(100))

# 4) Observer: notify subscribers of changes
class EventBus:
    def __init__(self): self.subs = []
    def subscribe(self, fn): self.subs.append(fn)
    def publish(self, msg):
        for fn in self.subs: fn(msg)
bus = EventBus()
bus.subscribe(lambda m: print("email:", m))
bus.subscribe(lambda m: print("sms  :", m))
bus.publish("order shipped")

# 5) Iterator/Decorator/Context manager patterns are built into the language (Module 09)

True
square
100 80.0
email: order shipped
sms  : order shipped


## 11. Complete worked example — Employee payroll with polymorphism

In [15]:
from abc import ABC, abstractmethod

class Staff(ABC):
    _count = 0
    def __init__(self, name):
        self.name = name
        Staff._count += 1

    @abstractmethod
    def monthly_pay(self): ...

    def __str__(self):
        return f"{type(self).__name__:10} {self.name:8} {self.monthly_pay():>10,.2f}"

    def __lt__(self, other): return self.monthly_pay() < other.monthly_pay()

    @classmethod
    def headcount(cls): return cls._count

class Salaried(Staff):
    def __init__(self, name, annual): super().__init__(name); self.annual = annual
    def monthly_pay(self): return self.annual / 12

class Hourly(Staff):
    def __init__(self, name, rate, hours): super().__init__(name); self.rate, self.hours = rate, hours
    def monthly_pay(self):
        overtime = max(0, self.hours - 160)
        return self.rate * (self.hours - overtime) + self.rate * 1.5 * overtime

class Intern(Staff):
    def monthly_pay(self): return 1500

team = [Salaried("Asha", 90000), Hourly("Ravi", 20, 170), Intern("Tom")]
for member in sorted(team, reverse=True):
    print(member)
print("Total payroll:", sum(m.monthly_pay() for m in team), "| headcount:", Staff.headcount())

Salaried   Asha       7,500.00
Hourly     Ravi       3,500.00
Intern     Tom        1,500.00
Total payroll: 12500.0 | headcount: 3


---
# 🕵️ Hidden Facts & Pro Tricks

In [16]:
# 1. super() works without arguments only inside methods; explicit form: super(Child, self)
# 2. Cooperative __init__ with **kwargs in multiple inheritance
class Base:
    def __init__(self, **kw): print("Base", kw)
class M1(Base):
    def __init__(self, a=None, **kw): print("M1", a); super().__init__(**kw)
class M2(Base):
    def __init__(self, b=None, **kw): print("M2", b); super().__init__(**kw)
class Both(M1, M2): pass
Both(a=1, b=2)

# 3. Overriding without calling super is legal - but check you really want to skip the parent
# 4. Prevent overriding by convention: name-mangle (__method), or use `typing.final`
from typing import final
class Locked:
    @final
    def critical(self): return "don't override me"

# 5. Class attribute overriding in subclasses = cheap configuration
class Base2:
    greeting = "Hello"
    def greet(self): return f"{self.greeting}, world"
class Fr(Base2): greeting = "Bonjour"
print(Base2().greet(), "|", Fr().greet())

# 6. __init_subclass__ + registry, __subclasses__ to discover children
print([c.__name__ for c in Shape.__subclasses__()])

# 7. Abstract properties & class methods are possible; ABCs can also "register" virtual subclasses
from collections.abc import Sequence, Mapping, Iterable
print(isinstance([], Sequence), isinstance({}, Mapping), isinstance(iter([]), Iterable))

# 8. Diamond: object is the ultimate parent of everything
print(int.__mro__, Duck.__mro__[-1])

# 9. Mocking is easy because of duck typing (tests can pass fake objects)
class FakeDB:
    def query(self, q): return [("fake", 1)]
def report(db): return len(db.query("select *"))
print(report(FakeDB()))

# 10. Data-driven polymorphism: dict of functions instead of if/elif chains on type
handlers = {int: lambda x: x * 2, str: lambda x: x.upper()}
print([handlers[type(v)](v) for v in (3, "a")])

# 11. Use enum + match for state machines; dataclass(slots=True) for lean data classes
# 12. `object.__new__` vs `__init__`: __new__ creates, __init__ configures (immutable subclasses need __new__)
class UpperStr(str):
    def __new__(cls, s): return super().__new__(cls, s.upper())
print(UpperStr("hey"))

M1 1
M2 2
Base {}
Hello, world | Bonjour, world
['Rect', 'Circle']
True True True
(<class 'int'>, <class 'object'>) <class 'object'>
1
[6, 'A']
HEY


**Hidden fact:** C3 linearization was borrowed from the Dylan language and made default in Python 2.3. Before that, multiple inheritance could silently pick the "wrong" method.

**Hidden fact:** You can inherit from built-ins: `class MyList(list)`, `class Stack(list)`, `class AttrDict(dict)`. But for wrapping behaviour, subclass `collections.UserList`/`UserDict` — built-in C methods sometimes bypass your overrides!

---
# 📝 Exam Corner

**Theory**
1. Define inheritance; explain its types with diagrams and examples.
2. What is `super()`? Why use it?
3. What is method overriding? Overriding vs overloading.
4. Explain MRO and the diamond problem; how does Python solve it?
5. Explain polymorphism (compile-time is n/a in Python; run-time & duck typing).
6. What is an abstract class? How do you create one? Can it be instantiated?
7. Inheritance vs composition.
8. What is `isinstance()`? `issubclass()`?

**Output prediction**

In [17]:
class A:
    def __init__(self): print("A init")
    def hi(self): print("A hi")
class B(A):
    def __init__(self):
        super().__init__()
        print("B init")
    def hi(self):
        print("B hi")
        super().hi()
class C(B):
    pass

C().hi()
print(C.__mro__[1].__name__, isinstance(C(), A))

class P:
    def f(self): return "P.f"
    def g(self): return self.f()
class Q(P):
    def f(self): return "Q.f"
print(Q().g())

A init
B init
B hi
A hi
A init
B init
B True
Q.f


<details><summary>💡 Answers</summary>

```
A init
B init
B hi
A hi
A init
B init
B True
Q.f
```
(The 2nd `C()` inside `print(...)` also runs the constructors, printing "A init / B init" before the print line shows `B True`.) `Q().g()` calls `self.f()` → looked up on the actual object (a `Q`) → `Q.f` — that's polymorphism.
</details>

---
## ✅ Summary
- `class Child(Parent)`; override to specialise; `super()` cooperates via the MRO.
- Polymorphism = same interface, different behaviour (overriding + duck typing).
- Abstract classes (`ABC` + `@abstractmethod`) enforce contracts.
- Prefer composition for "has-a"; use mixins for small reusable behaviours.

➡️ **Next:** Module 15 — Recursion, Algorithms & Complexity.